# Fine-tune BERT for classification

A university IT helpdesk wants every incoming ticket sent to the Accounts, Wi-Fi or Printing team
automatically. You will fine-tune a small pretrained BERT on 72 labelled tickets with the Hugging Face
`Trainer`, test it on the 20 tickets you labelled by hand in the lesson, then break the setup on purpose twice.

About 5 minutes on a free Colab CPU runtime. No GPU, account or key is needed. Run the cells from top to bottom.

## 1. Install the library versions we used

Colab already has PyTorch and Transformers. This cell pins Transformers and Accelerate to the versions the
lesson was run with, so your printed output looks like ours. It prints nothing when it works.

In [1]:
%pip install -q transformers==5.23.0 accelerate==1.15.0

Note: you may need to restart the kernel to use updated packages.


## 2. The tickets

The same tickets as in the lesson, written by the lesson author. Labels are numbers:
0 = accounts, 1 = wifi, 2 = printing. `TRAIN` has 72 tickets to learn from. `TEST` has your 20 tickets;
the model never trains on them.

**Look for:** 72 training tickets, 24 per team, and 20 test tickets.

In [2]:
TEAMS = ["accounts", "wifi", "printing"]

TRAIN = [
    # accounts (0)
    ("I forgot my university password and the reset email never arrived.", 0),
    ("My student account is locked after too many login attempts.", 0),
    ("How do I change the phone number used for two-factor codes?", 0),
    ("The portal says my username does not exist but I enrolled last week.", 0),
    ("I got a new phone and the authenticator app no longer shows my codes.", 0),
    ("Please give me access to the shared drive for the chemistry department.", 0),
    ("My email account was disabled, I am still a registered student.", 0),
    ("I cannot sign in to the library website with my student ID.", 0),
    ("My name is spelled wrong on my staff account, can you fix it?", 0),
    ("The password reset page keeps saying the link has expired.", 0),
    ("I need a guest account for a visiting professor for two weeks.", 0),
    ("Single sign-on sends me back to the login page in a loop.", 0),
    ("My account still shows my old department after I moved to physics.", 0),
    ("I want to set up the security questions for my account.", 0),
    ("Someone may know my password, please lock my account now.", 0),
    ("The learning platform says I am not allowed to view my own course.", 0),
    ("How long before a new staff member's login starts working?", 0),
    ("I keep getting 'invalid credentials' even with the right password.", 0),
    ("Please remove a former student's access to our research group folder.", 0),
    ("My verification code arrives by text but the site rejects it.", 0),
    ("I graduated, can I keep my university email address?", 0),
    ("The system asks me to change my password every day.", 0),
    ("I lost the backup codes for my two-step login.", 0),
    ("My alumni account login does not work since the update.", 0),
    # wifi (1)
    ("The wireless network in the library keeps dropping every few minutes.", 1),
    ("My laptop connects to the campus Wi-Fi but no web pages load.", 1),
    ("There is no Wi-Fi signal at all in room B204.", 1),
    ("The eduroam profile on my phone stopped working after an update.", 1),
    ("Internet is extremely slow in the dormitory in the evenings.", 1),
    ("My tablet cannot find the guest network in the main hall.", 1),
    ("The Wi-Fi certificate warning appears every time I connect.", 1),
    ("Video calls freeze in the lecture theatre because the connection is weak.", 1),
    ("The access point on the third floor has a blinking light and nobody can connect.", 1),
    ("My game console in the residence cannot join the wireless network.", 1),
    ("I get 'no internet, secured' on my laptop in the cafeteria.", 1),
    ("The network disconnects whenever I walk between buildings.", 1),
    ("Downloads stop halfway on the campus wireless.", 1),
    ("Our lab's wireless router seems to be switched off.", 1),
    ("Students in the exam hall cannot get online for the online test.", 1),
    ("My phone says the saved network is out of range in the gym.", 1),
    ("The connection in the student union drops when many people are there.", 1),
    ("Which network should visitors use to get online?", 1),
    ("The wireless works on my phone but not on my new laptop.", 1),
    ("Websites time out on the campus network since this morning.", 1),
    ("Is there a Wi-Fi outage in the engineering building?", 1),
    ("The signal is strong but the speed test shows almost nothing.", 1),
    ("I cannot connect my smart TV in the residence to the network.", 1),
    ("The hotspot in the seminar room keeps asking me to log in again.", 1),
    # printing (2)
    ("The printer on the second floor of the library is jammed.", 2),
    ("My print job is stuck in the queue and never comes out.", 2),
    ("The printer says it is out of toner.", 2),
    ("How do I add credit to my printing balance?", 2),
    ("I was charged twice for one print job.", 2),
    ("The copier in the staff room prints everything with grey streaks.", 2),
    ("I cannot find the student printer in the list on my laptop.", 2),
    ("The printer will only print single-sided even when I choose double.", 2),
    ("There is no paper in the printers in the computer lab.", 2),
    ("My poster came out cut off at the edges.", 2),
    ("The release station does not recognise my student card.", 2),
    ("Scanned documents are not arriving in my email from the copier.", 2),
    ("The colour printer prints everything in black and white.", 2),
    ("The printer in room C110 shows an error code and stops.", 2),
    ("Pages come out blank from the printer near the entrance.", 2),
    ("Can I print from my phone to the library printers?", 2),
    ("My thesis printed with the wrong margins.", 2),
    ("The printer keeps printing the same page over and over.", 2),
    ("The scanner glass is cracked on the third-floor copier.", 2),
    ("How much does it cost to print in colour?", 2),
    ("The print driver fails to install on my Mac.", 2),
    ("My printouts are smudged and the ink is wet.", 2),
    ("The printer queue shows jobs from other people with my name.", 2),
    ("The large-format printer for posters is offline.", 2),
]

# The twenty tickets the learner labels by hand. Some are deliberately close to two teams.
TEST = [
    ("I typed my password wrong and now I am locked out of everything.", 0),
    ("The Wi-Fi in the reading room disconnects every ten minutes.", 1),
    ("The printer in the history building is out of paper again.", 2),
    ("I cannot log in to the campus Wi-Fi since I changed my password.", 0),
    ("My print credit did not go up after I paid.", 2),
    ("The internet is very slow in residence block D.", 1),
    ("Please create an account for our new lab technician.", 0),
    ("The library copier jams on every second page.", 2),
    ("My laptop cannot see any wireless networks in the sports hall.", 1),
    ("The two-factor prompt never appears on my phone.", 0),
    ("I sent my essay to print but nothing came out.", 2),
    ("Online lectures keep buffering when I am in the dorm.", 1),
    ("The printer asks me to sign in and says my username is unknown.", 0),
    ("How do I get my account back after it expired?", 0),
    ("The scanner sends empty files to my email.", 2),
    ("There is no connection in the new seminar rooms.", 1),
    ("The printer prints faint, hard-to-read text.", 2),
    ("My username is not accepted on the guest Wi-Fi page.", 0),
    ("Web pages load halfway and then stop in the canteen.", 1),
    ("Staples got stuck inside the copier in room A12.", 2),
]


print("training tickets:", len(TRAIN))
print("per team:", {t: sum(1 for _, l in TRAIN if l == k) for k, t in enumerate(TEAMS)})
print("test tickets:", len(TEST))

training tickets: 72
per team: {'accounts': 24, 'wifi': 24, 'printing': 24}
test tickets: 20


## 3. The tokenizer and the datasets

The model reads numbers, not letters. The checkpoint's own tokenizer turns each ticket into token ids. The
small `Tickets` class hands the `Trainer` one ticket at a time: its token ids and its label.

The model is BERT-Mini (4 layers, 256 wide, about 45 MB), downloaded from the Hugging Face Hub at a fixed
revision. You may see a warning about `HF_TOKEN`: ignore it, no account is needed.

**Look for:** the first ticket split into word pieces, starting with `[CLS]` and ending with `[SEP]`.

In [3]:
import numpy as np
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          set_seed)

CHECKPOINT = "google/bert_uncased_L-4_H-256_A-4"
REVISION = "387825ce42dbb39b87911cdf8e383ee3b25184f8"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, revision=REVISION)


class Tickets(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.enc = tokenizer([t for t, _ in rows], truncation=True)
        self.labels = [l for _, l in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i],
                "token_type_ids": self.enc["token_type_ids"][i], "labels": self.labels[i]}


train_set, test_set = Tickets(TRAIN), Tickets(TEST)
print("ticket:", TEST[0][0])
print("tokens:", tokenizer.convert_ids_to_tokens(test_set[0]["input_ids"]))

ticket: I typed my password wrong and now I am locked out of everything.
tokens: ['[CLS]', 'i', 'typed', 'my', 'password', 'wrong', 'and', 'now', 'i', 'am', 'locked', 'out', 'of', 'everything', '.', '[SEP]']


## 4. Build the model and the Trainer

This is the lesson's code, wrapped in a function so you can rebuild it later with other settings.
`num_labels` says how many outputs the new head gets. `torch.manual_seed(0)` makes the random head the same
on every run. `disable_tqdm=True` makes the `Trainer` print one plain line per log instead of a progress bar.

While loading, the library prints a **load report**. **Look for:** `classifier.weight` and `classifier.bias`
marked MISSING (a new head, started at random), and the `cls.…` pretraining heads marked UNEXPECTED (thrown
away). Then the parameter count: 771 of about 11.2 million weights are in the new head.

In [4]:
def accuracy(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": float((np.argmax(logits, axis=1) == labels).mean())}


def build(num_labels=3, learning_rate=1e-4, epochs=8, name="good"):
    id2label = dict(enumerate(TEAMS[:num_labels]))
    label2id = {v: k for k, v in id2label.items()}
    set_seed(0)
    torch.manual_seed(0)
    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT, revision=REVISION, num_labels=num_labels, id2label=id2label, label2id=label2id)
    args = TrainingArguments(output_dir=f"out-{name}", learning_rate=learning_rate,
        num_train_epochs=epochs, per_device_train_batch_size=8, eval_strategy="epoch",
        logging_strategy="epoch", save_strategy="no", seed=0, report_to="none", use_cpu=True,
        disable_tqdm=True)
    return Trainer(model=model, args=args, train_dataset=train_set, eval_dataset=test_set,
        processing_class=tokenizer, compute_metrics=accuracy)


trainer = build()
model = trainer.model
print("all weights:     ", sum(p.numel() for p in model.parameters()))
print("weights in head: ", sum(p.numel() for n, p in model.named_parameters() if n.startswith("classifier")))

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

all weights:      11171331
weights in head:  771


## 5. Before training: route your 20 tickets

The body has read lots of English, but the head is random. This cell asks the untrained model for a team for
each of your tickets, with its probability for each team.

**Look for:** Printing chosen for every ticket, each time with a low probability (about 0.41 to 0.46), and an
accuracy of 0.35. That is just the share of Printing tickets (7 of 20).

In [5]:
def route(trainer, title):
    out = trainer.predict(test_set)
    probs = torch.softmax(torch.tensor(out.predictions), dim=1).numpy()
    names = trainer.model.config.id2label
    print(title)
    print(f"{'#':>2}  {'helpdesk':9} {'model':9} probabilities (accounts, wifi, printing)")
    for i, ((text, gold), p) in enumerate(zip(TEST, probs)):
        pred = names[int(p.argmax())]
        mark = "" if pred == TEAMS[gold] else "   <- wrong"
        print(f"{i + 1:>2}  {TEAMS[gold]:9} {pred:9} {[round(float(x), 3) for x in p]}{mark}")
    print("accuracy:", out.metrics["test_accuracy"])


route(trainer, "Before training")

Before training
 #  helpdesk  model     probabilities (accounts, wifi, printing)
 1  accounts  printing  [0.269, 0.281, 0.449]   <- wrong
 2  wifi      printing  [0.242, 0.308, 0.45]   <- wrong
 3  printing  printing  [0.247, 0.323, 0.431]
 4  accounts  printing  [0.245, 0.332, 0.423]   <- wrong
 5  printing  printing  [0.268, 0.303, 0.429]
 6  wifi      printing  [0.266, 0.313, 0.42]   <- wrong
 7  accounts  printing  [0.287, 0.294, 0.419]   <- wrong
 8  printing  printing  [0.258, 0.319, 0.423]
 9  wifi      printing  [0.274, 0.315, 0.411]   <- wrong
10  accounts  printing  [0.27, 0.271, 0.46]   <- wrong
11  printing  printing  [0.27, 0.282, 0.448]
12  wifi      printing  [0.249, 0.305, 0.446]   <- wrong
13  accounts  printing  [0.26, 0.297, 0.444]   <- wrong
14  accounts  printing  [0.292, 0.277, 0.431]   <- wrong
15  printing  printing  [0.267, 0.271, 0.462]
16  wifi      printing  [0.254, 0.315, 0.431]   <- wrong
17  printing  printing  [0.262, 0.281, 0.457]
18  accounts  printing

## 6. Train

`trainer.train()` runs the whole loop: batches of 8, the loss, the backward pass, the optimizer, and an
evaluation on your 20 tickets after each epoch. 8 epochs of 9 batches. On a free Colab CPU it takes under a
minute.

Each epoch prints two lines: one with the training `loss`, one with `eval_accuracy` on your held-out tickets.

**Look for:** the loss starts near 1.1 (ln 3, the loss of guessing evenly) and falls every epoch, to about
0.38. `eval_accuracy` reaches 0.9 at epoch 2 and then stays about there (0.95 once, at epoch 4).

In [6]:
trainer.train()

{'loss': '1.12', 'grad_norm': '7.618', 'learning_rate': '8.889e-05', 'epoch': '1'}
{'eval_loss': '1.005', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.7', 'eval_runtime': '0.0375', 'eval_samples_per_second': '533.1', 'eval_steps_per_second': '79.97', 'epoch': '1'}


{'loss': '0.9748', 'grad_norm': '6.489', 'learning_rate': '7.639e-05', 'epoch': '2'}
{'eval_loss': '0.8979', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.0388', 'eval_samples_per_second': '515.2', 'eval_steps_per_second': '77.28', 'epoch': '2'}


{'loss': '0.8404', 'grad_norm': '4.274', 'learning_rate': '6.389e-05', 'epoch': '3'}
{'eval_loss': '0.7914', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.042', 'eval_samples_per_second': '476', 'eval_steps_per_second': '71.4', 'epoch': '3'}


{'loss': '0.7174', 'grad_norm': '4.286', 'learning_rate': '5.139e-05', 'epoch': '4'}
{'eval_loss': '0.6776', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.95', 'eval_runtime': '0.0494', 'eval_samples_per_second': '404.6', 'eval_steps_per_second': '60.69', 'epoch': '4'}


{'loss': '0.5734', 'grad_norm': '5.452', 'learning_rate': '3.889e-05', 'epoch': '5'}
{'eval_loss': '0.5824', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.0402', 'eval_samples_per_second': '497.4', 'eval_steps_per_second': '74.6', 'epoch': '5'}


{'loss': '0.4808', 'grad_norm': '3.192', 'learning_rate': '2.639e-05', 'epoch': '6'}
{'eval_loss': '0.5291', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.04', 'eval_samples_per_second': '500.3', 'eval_steps_per_second': '75.04', 'epoch': '6'}


{'loss': '0.411', 'grad_norm': '3.271', 'learning_rate': '1.389e-05', 'epoch': '7'}
{'eval_loss': '0.4956', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.0459', 'eval_samples_per_second': '435.4', 'eval_steps_per_second': '65.31', 'epoch': '7'}


{'loss': '0.3798', 'grad_norm': '2.978', 'learning_rate': '1.389e-06', 'epoch': '8'}
{'eval_loss': '0.492', 'eval_model_preparation_time': '0.0022', 'eval_accuracy': '0.9', 'eval_runtime': '0.039', 'eval_samples_per_second': '513.4', 'eval_steps_per_second': '77.01', 'epoch': '8'}
{'train_runtime': '6.194', 'train_samples_per_second': '92.99', 'train_steps_per_second': '11.62', 'train_loss': '0.6872', 'epoch': '8'}


TrainOutput(global_step=72, training_loss=0.6871958805455102, metrics={'train_runtime': 6.1941, 'train_samples_per_second': 92.991, 'train_steps_per_second': 11.624, 'train_loss': 0.6871958805455102, 'epoch': 8.0})

## 7. The training log as a table

The same numbers as above, one row per epoch, taken from `trainer.state.log_history`.

**Look for:** the training loss keeps falling, but held-out accuracy stops rising after epoch 2. More epochs
bought nothing on tickets the model never saw.

In [7]:
def epoch_table(trainer):
    loss = {round(h["epoch"]): h["loss"] for h in trainer.state.log_history if "loss" in h}
    acc = {round(h["epoch"]): h["eval_accuracy"] for h in trainer.state.log_history if "eval_accuracy" in h}
    print("epoch  training loss  held-out accuracy")
    for e in sorted(loss):
        print(f"{e:>5}  {loss[e]:>13.4f}  {acc[e]:>17.2f}")


epoch_table(trainer)

epoch  training loss  held-out accuracy
    1         1.1199               0.70
    2         0.9748               0.90
    3         0.8404               0.90
    4         0.7174               0.95
    5         0.5734               0.90
    6         0.4808               0.90
    7         0.4110               0.90
    8         0.3798               0.90


## 8. After training: route your 20 tickets again

**Look for:** accuracy 0.9, so 2 tickets go to the wrong team: ticket 8 (a copier jam, sent to wifi) and
ticket 13 (an unknown username at a printer, sent to printing). Both mistakes are made with low confidence,
about 0.5.

In [8]:
route(trainer, "After training")

After training
 #  helpdesk  model     probabilities (accounts, wifi, printing)
 1  accounts  accounts  [0.774, 0.105, 0.122]
 2  wifi      wifi      [0.105, 0.67, 0.226]
 3  printing  printing  [0.086, 0.184, 0.73]
 4  accounts  accounts  [0.675, 0.257, 0.068]
 5  printing  printing  [0.207, 0.122, 0.671]
 6  wifi      wifi      [0.101, 0.66, 0.239]
 7  accounts  accounts  [0.719, 0.148, 0.133]
 8  printing  wifi      [0.12, 0.507, 0.373]   <- wrong
 9  wifi      wifi      [0.128, 0.698, 0.174]
10  accounts  accounts  [0.447, 0.383, 0.17]
11  printing  printing  [0.224, 0.119, 0.656]
12  wifi      wifi      [0.136, 0.611, 0.253]
13  accounts  printing  [0.355, 0.141, 0.504]   <- wrong
14  accounts  accounts  [0.761, 0.107, 0.132]
15  printing  printing  [0.223, 0.16, 0.617]
16  wifi      wifi      [0.109, 0.67, 0.221]
17  printing  printing  [0.097, 0.148, 0.756]
18  accounts  accounts  [0.727, 0.159, 0.114]
19  wifi      wifi      [0.114, 0.503, 0.383]
20  printing  printing  [0.132,

## 9. Broken on purpose: the wrong label count

A colleague kept `num_labels=2` from a two-label example. The labels are 0, 1 and 2, but the head now has only
two outputs. The cell catches the error so the notebook keeps running.

**Look for:** the load report shows the new head again, then the run stops on the first batch with
`IndexError: Target 2 is out of bounds.`

In [9]:
two = build(num_labels=2, name="two-labels")
try:
    two.train()
except IndexError as err:
    print("The run stopped with:", type(err).__name__ + ":", err)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

The run stopped with: IndexError: Target 2 is out of bounds.


## 10. Broken on purpose: a learning rate far too big

Same code, `learning_rate=1e-2` instead of `1e-4`: a hundred times bigger. No crash this time.

**Look for:** the training loss never goes below about 1.15, at or above the guessing line ln 3 ≈ 1.099.
Held-out accuracy stays between 0.3 and 0.35. At the end, every ticket gets exactly the same probabilities,
`[0.35, 0.399, 0.25]`, so the model says wifi to all 20 tickets.

In [10]:
high = build(learning_rate=1e-2, name="lr-too-high")
high.train()
print()
epoch_table(high)
print()
route(high, "After training with learning_rate=1e-2")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

{'loss': '1.864', 'grad_norm': '5.716', 'learning_rate': '0.008889', 'epoch': '1'}
{'eval_loss': '1.967', 'eval_accuracy': '0.35', 'eval_runtime': '0.0708', 'eval_samples_per_second': '282.5', 'eval_steps_per_second': '42.37', 'epoch': '1'}


{'loss': '1.466', 'grad_norm': '6.635', 'learning_rate': '0.007639', 'epoch': '2'}
{'eval_loss': '1.695', 'eval_accuracy': '0.35', 'eval_runtime': '0.101', 'eval_samples_per_second': '198.1', 'eval_steps_per_second': '29.71', 'epoch': '2'}


{'loss': '1.533', 'grad_norm': '6.284', 'learning_rate': '0.006389', 'epoch': '3'}
{'eval_loss': '1.671', 'eval_accuracy': '0.3', 'eval_runtime': '0.0353', 'eval_samples_per_second': '567.3', 'eval_steps_per_second': '85.09', 'epoch': '3'}


{'loss': '1.36', 'grad_norm': '2.547', 'learning_rate': '0.005139', 'epoch': '4'}
{'eval_loss': '1.179', 'eval_accuracy': '0.35', 'eval_runtime': '0.0264', 'eval_samples_per_second': '758.6', 'eval_steps_per_second': '113.8', 'epoch': '4'}


{'loss': '1.278', 'grad_norm': '11.8', 'learning_rate': '0.003889', 'epoch': '5'}
{'eval_loss': '1.453', 'eval_accuracy': '0.3', 'eval_runtime': '0.0212', 'eval_samples_per_second': '945.6', 'eval_steps_per_second': '141.8', 'epoch': '5'}


{'loss': '1.228', 'grad_norm': '6.253', 'learning_rate': '0.002639', 'epoch': '6'}
{'eval_loss': '1.129', 'eval_accuracy': '0.35', 'eval_runtime': '0.0227', 'eval_samples_per_second': '882.3', 'eval_steps_per_second': '132.3', 'epoch': '6'}


{'loss': '1.153', 'grad_norm': '3.212', 'learning_rate': '0.001389', 'epoch': '7'}
{'eval_loss': '1.128', 'eval_accuracy': '0.3', 'eval_runtime': '0.0278', 'eval_samples_per_second': '719.9', 'eval_steps_per_second': '108', 'epoch': '7'}


{'loss': '1.176', 'grad_norm': '3.501', 'learning_rate': '0.0001389', 'epoch': '8'}
{'eval_loss': '1.127', 'eval_accuracy': '0.3', 'eval_runtime': '0.0409', 'eval_samples_per_second': '488.5', 'eval_steps_per_second': '73.28', 'epoch': '8'}
{'train_runtime': '6.09', 'train_samples_per_second': '94.57', 'train_steps_per_second': '11.82', 'train_loss': '1.382', 'epoch': '8'}

epoch  training loss  held-out accuracy
    1         1.8640               0.35
    2         1.4664               0.35
    3         1.5331               0.30
    4         1.3600               0.35
    5         1.2775               0.30
    6         1.2277               0.35
    7         1.1529               0.30
    8         1.1763               0.30

After training with learning_rate=1e-2
 #  helpdesk  model     probabilities (accounts, wifi, printing)
 1  accounts  wifi      [0.35, 0.399, 0.25]   <- wrong
 2  wifi      wifi      [0.35, 0.399, 0.25]
 3  printing  wifi      [0.35, 0.399, 0.25]   <- wrong
 4  

## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it and run the cell again.

### Experiment 1: route a ticket of your own

This uses the model you trained in step 6. Write any helpdesk ticket on the marked line.

**What should change:** the team and the three probabilities. The ticket as written mentions a printer, but
the fault is a password, and the model sends it to accounts at about 0.7. Now try a ticket with no clear fault
that mixes two teams' words, such as "The printer next to the Wi-Fi router is slow", and watch whether the
three numbers move closer together. Close numbers mean the model is unsure.

In [11]:
ticket = "I cannot log in to the printer website because my password expired."  # <- change this

enc = tokenizer(ticket, return_tensors="pt")
trainer.model.eval()
with torch.no_grad():
    p = torch.softmax(trainer.model(**enc).logits, dim=1)[0].numpy()
for k, team in enumerate(TEAMS):
    print(f"{team:9} {p[k]:.3f}")
print("team:", TEAMS[int(p.argmax())])

accounts  0.695
wifi      0.108
printing  0.197
team: accounts


### Experiment 2: fewer epochs

Train again with `epochs=2` instead of 8. The learning rate still shrinks to zero by the end, just faster.

**What should change:** the loss only falls to about 1.0, against 0.38 after 8 epochs. Held-out accuracy is
still 0.9 in our run, but the model is much less sure. Its highest probability per ticket is only about 0.35
to 0.42, close to an even guess (0.33), against 0.45 to 0.77 after 8 epochs. It also misses different tickets
(3 and 8). Accuracy alone hides how unsure it is.

In [12]:
short = build(epochs=2, name="short")  # <- change this (try epochs=4)
short.train()
print()
epoch_table(short)
print()
route(short, "After 2 epochs")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

{'loss': '1.098', 'grad_norm': '7.762', 'learning_rate': '5.556e-05', 'epoch': '1'}
{'eval_loss': '0.9987', 'eval_accuracy': '0.9', 'eval_runtime': '0.0272', 'eval_samples_per_second': '734.9', 'eval_steps_per_second': '110.2', 'epoch': '1'}


{'loss': '0.9979', 'grad_norm': '7.241', 'learning_rate': '5.556e-06', 'epoch': '2'}
{'eval_loss': '0.9649', 'eval_accuracy': '0.9', 'eval_runtime': '0.0404', 'eval_samples_per_second': '495.2', 'eval_steps_per_second': '74.28', 'epoch': '2'}
{'train_runtime': '1.49', 'train_samples_per_second': '96.68', 'train_steps_per_second': '12.09', 'train_loss': '1.048', 'epoch': '2'}

epoch  training loss  held-out accuracy
    1         1.0975               0.90
    2         0.9979               0.90

After 2 epochs
 #  helpdesk  model     probabilities (accounts, wifi, printing)
 1  accounts  accounts  [0.385, 0.28, 0.335]
 2  wifi      wifi      [0.279, 0.411, 0.31]
 3  printing  wifi      [0.273, 0.367, 0.361]   <- wrong
 4  accounts  accounts  [0.388, 0.334, 0.278]
 5  printing  printing  [0.362, 0.272, 0.366]
 6  wifi      wifi      [0.285, 0.42, 0.294]
 7  accounts  accounts  [0.393, 0.33, 0.277]
 8  printing  wifi      [0.293, 0.391, 0.315]   <- wrong
 9  wifi      wifi      [0.334, 0.

### Experiment 3: a smaller learning rate

The model card for these small BERT models tested learning rates from 3e-4 down to 3e-5. Try the low end,
`3e-5`.

**What should change:** the loss falls much more slowly and ends near 0.94, against 0.38 with 1e-4, because
each weight update is about three times smaller. Held-out accuracy also climbs more slowly: 0.45 after epoch 1,
and 0.9 only from epoch 6. Too small wastes epochs; too big (step 10) never learns.

In [13]:
slow = build(learning_rate=3e-5, name="slow")  # <- change this (try 3e-4)
slow.train()
print()
epoch_table(slow)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

{'loss': '1.107', 'grad_norm': '6.677', 'learning_rate': '2.667e-05', 'epoch': '1'}
{'eval_loss': '1.048', 'eval_accuracy': '0.45', 'eval_runtime': '0.0409', 'eval_samples_per_second': '488.9', 'eval_steps_per_second': '73.34', 'epoch': '1'}


{'loss': '1.053', 'grad_norm': '6.844', 'learning_rate': '2.292e-05', 'epoch': '2'}
{'eval_loss': '1.008', 'eval_accuracy': '0.7', 'eval_runtime': '0.0257', 'eval_samples_per_second': '777.5', 'eval_steps_per_second': '116.6', 'epoch': '2'}


{'loss': '1.025', 'grad_norm': '3.143', 'learning_rate': '1.917e-05', 'epoch': '3'}
{'eval_loss': '0.9745', 'eval_accuracy': '0.85', 'eval_runtime': '0.0237', 'eval_samples_per_second': '845', 'eval_steps_per_second': '126.7', 'epoch': '3'}


{'loss': '0.9903', 'grad_norm': '3.421', 'learning_rate': '1.542e-05', 'epoch': '4'}
{'eval_loss': '0.9545', 'eval_accuracy': '0.8', 'eval_runtime': '0.0318', 'eval_samples_per_second': '628.2', 'eval_steps_per_second': '94.22', 'epoch': '4'}


{'loss': '0.9506', 'grad_norm': '4.869', 'learning_rate': '1.167e-05', 'epoch': '5'}
{'eval_loss': '0.9394', 'eval_accuracy': '0.8', 'eval_runtime': '0.0454', 'eval_samples_per_second': '440.2', 'eval_steps_per_second': '66.03', 'epoch': '5'}


{'loss': '0.9451', 'grad_norm': '3.755', 'learning_rate': '7.917e-06', 'epoch': '6'}
{'eval_loss': '0.9217', 'eval_accuracy': '0.9', 'eval_runtime': '0.0454', 'eval_samples_per_second': '440.1', 'eval_steps_per_second': '66.02', 'epoch': '6'}


{'loss': '0.9352', 'grad_norm': '3.084', 'learning_rate': '4.167e-06', 'epoch': '7'}
{'eval_loss': '0.9122', 'eval_accuracy': '0.9', 'eval_runtime': '0.0358', 'eval_samples_per_second': '558.4', 'eval_steps_per_second': '83.76', 'epoch': '7'}


{'loss': '0.9378', 'grad_norm': '3.656', 'learning_rate': '4.167e-07', 'epoch': '8'}
{'eval_loss': '0.9092', 'eval_accuracy': '0.9', 'eval_runtime': '0.0194', 'eval_samples_per_second': '1029', 'eval_steps_per_second': '154.3', 'epoch': '8'}
{'train_runtime': '5.163', 'train_samples_per_second': '111.6', 'train_steps_per_second': '13.95', 'train_loss': '0.993', 'epoch': '8'}

epoch  training loss  held-out accuracy
    1         1.1073               0.45
    2         1.0530               0.70
    3         1.0250               0.85
    4         0.9903               0.80
    5         0.9506               0.80
    6         0.9451               0.90
    7         0.9352               0.90
    8         0.9378               0.90


## What you should see

- The load report lists `classifier.weight` and `classifier.bias` as MISSING (newly initialized).
- Before training, accuracy on the 20 held-out tickets is low, near what one team for everything gives (ours: 35%).
- The training loss starts near 1.1 and falls every epoch (ours: 1.12 to 0.3798).
- Held-out accuracy reaches about 0.9 in the first few epochs and then stays about there (ours: from epoch 2).
- With `num_labels=2` the run stops with `IndexError: Target 2 is out of bounds.`
- With `learning_rate=1e-2` the loss never goes below about 1.1, and every ticket gets the same team.